In [ ]:
# Make repo root importable for this session (zero packaging)
import sys, pathlib
repo_root = pathlib.Path.cwd().parent if (pathlib.Path.cwd().name == "dev_notebooks") else pathlib.Path.cwd()
sys.path.insert(0, str(repo_root))
%load_ext autoreload
%autoreload 2

In [ ]:
!ls

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import os
import argparse
import soundfile as sf
from IPython.display import Audio
import time
from pathlib import Path 

from transformers import EncodecModel

In [ ]:
# Local imports from your project structure
from rnencodec.model.gru_audio_model import RNN, GRUModelConfig
from rnencodec.utils.utils import multi_linspace, steps, plot_condition_tensor
from rnencodec.audioDataLoader.audio_dataset import LatentDatasetConfig

#from rnencodec.inference_orig import run_inference
from rnencodec.generator import RNNGenerator

In [ ]:
# Point run dir to the folder that Train.ipynb created for its run.

# good one ... run_directory = str(Path('./output/20250811_135640')) # 'Path to the directory of the saved run.'
#run_directory = str(Path('./output/20250828_173834_encodectest_DSWind')) # 'Path to the directory of the saved run.'
#run_directory = str(Path('./output/20250905_130844_encodectest_DSWind_splitseqs')) # 'Path to the directory of the saved run.'
#run_directory = str(Path('./output/20250906_152317_encodectest_DSBugs_splitseqs_sl125_3l_size64_nq4'))
#run_directory = str(Path('./output/20250906_170007_encodectest_DSPistons_splitseqs_sl125_3l_size64_nq4'))
#run_directory = str(Path('./output/20250907_113358_encodectest_TokWotalDuet_splitseqs_sl125_3l_size128_nq8_emb4.5'))
#run_directory = str(Path('../output_keep/20250926_220930_wild_WATER_CONTINUOUS'))
run_directory = str(Path('../artifacts/weights'))
run_directory = str(Path('../quickstart/output/2026-01-29_13:39:29_quickstart5'))

top_n = 24 #'Sample from the top N most likely outputs.'
temperature = .8 #'Controls the randomness of predictions.'
length_seconds =20.0 #'Length of the audio to generate in seconds.'

frame_rate=75
generation_length = int(length_seconds * frame_rate)

sr=24000

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
device = 'cpu'
device

In [ ]:
run_directory

In [ ]:
!ls $run_directory

In [ ]:
# -------     Load model     -----------#

config_path = os.path.join(run_directory, "config_v2.pt")
#checkpoint_path = os.path.join(run_directory, "checkpoints", "checkpoint_225.pt") # "last_checkpoint.pt") # "checkpoint_40.pt") # 
checkpoint_path = os.path.join(run_directory, "checkpoints/last_checkpoint.pt") # "last_checkpoint.pt") # "checkpoint_40.pt") # 

assert os.path.exists(run_directory), f"Run directory not found: {run_directory}"
assert os.path.exists(config_path), f"Config file not found: {config_path}"
assert os.path.exists(checkpoint_path), f"Checkpoint file not found: {checkpoint_path}"

saved_configs = torch.load(config_path, weights_only=False)

In [ ]:
saved_configs

In [ ]:
model_config = GRUModelConfig(**saved_configs['model_config']) 
data_config = LatentDatasetConfig(**saved_configs["data_config"])
clamp_val = data_config.clamp_val # need this to map between encodec latents and model input ranges

enc_model = EncodecModel.from_pretrained("facebook/encodec_24khz").to(device)
enc_model.eval()

model = RNN(model_config, enc_model).to(device)
checkpoint = torch.load(checkpoint_path, map_location="cpu")
state = checkpoint["model_state_dict"] if "model_state_dict" in checkpoint else checkpoint
model.load_state_dict(state, strict=False)  # False if your export is fp16; True if fp32
model.to(device).eval()



print("Model successfully loaded from checkpoint.")
print(f"Using device = {device}")
#clamp_val 

In [ ]:

num_cond_params = model_config.cond_size
cond_seq = torch.zeros(generation_length, num_cond_params)

# instID
# cond_seq[:, 0] = torch.FloatTensor(steps(np.array([0,1,0,0,0,0,0,0,0]), generation_length))
# cond_seq[:, 1] = torch.FloatTensor(steps(np.array([0,0,1,0,0,0,0,0,0]), generation_length))
# cond_seq[:, 2] = torch.FloatTensor(steps(np.array([0,0,0,1,0,0,0,0,0]), generation_length))
# cond_seq[:, 3] = torch.FloatTensor(steps(np.array([0,0,0,0,1,0,0,0,0]), generation_length))
# cond_seq[:, 4] = torch.FloatTensor(steps(np.array([0,0,0,0,0,1,0,0,0]), generation_length))
# cond_seq[:, 5] = torch.FloatTensor(steps(np.array([0,0,0,0,0,0,1,0,0]), generation_length))
# cond_seq[:, 6] = torch.FloatTensor(steps(np.array([0,0,0,0,0,0,0,1,0]), generation_length))

# param_1
cond_seq[:, 0] = .5


In [ ]:
#For nsynth.64.76_sm, the parameter 
#  Param1 - instID
#  Param2 - a (amplitude)
#  Param3 - p (pitch in [0,1], representing midi [64, 76]

plot_condition_tensor(cond_seq, frame_rate)

In [ ]:
cond_seq.shape

In [ ]:
chunksize=2
hopsize=2
bazrnn=RNNGenerator.from_checkpoint(checkpoint_path, model_config, data_config, enc_model, chunksize, hopsize) #, "sample", top_n, temperature)
#warm up prevents noise bursts at the begininng of the audio run
_ = bazrnn.warmup(cond_seq[0], 10)
start = time.perf_counter()
generated_audio=bazrnn.getNextAudioHop(cond_seq.to(device)) #length of cond_seq overrides hopsize
elapsed = time.perf_counter() - start
print(f"Generated { generated_audio.shape[0]} samples  in {elapsed:.2f}s.")

In [ ]:

# warmup_len = 1
# sd=.33 # to create data in [-1,1]
# warmup_sequence = torch.clamp(torch.randn(warmup_len, 128) * sd, -3*sd, 3*sd)    

# # want :    x.shape = torch.Size([100, 128]), and c.shape = torch.Size([genseqlen, 1])
# print(f'About to call inference with  warmup_sequence.shape = {warmup_sequence.shape}, and cond_seq.shape = {cond_seq.shape}')

# start = time.perf_counter()

# generated_audio, sr = run_inference(model, enc_model, cond_seq.to(device), warmup_sequence.to(device), clamp_val, top_n=top_n, temperature=temperature, include_warmup_audio=False)

# elapsed = time.perf_counter() - start
# print(f"Generated { generated_audio.shape[0]} samples at {sr} Hz in {elapsed:.2f}s.")

In [ ]:
t = np.arange(len(generated_audio)) / sr

#print(f"Saving waveform plot to {args.output_plot_path}")
plt.figure(figsize=(20, 5))
plt.plot(t,generated_audio)
plt.title("Generated Audio Waveform")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.grid()
#plt.savefig(args.output_plot_path)
#plt.close()

plt.show()

In [ ]:
Audio(generated_audio, rate=sr)

In [ ]:
#---------------------------------------------------  pitch glide  --------------------#

In [ ]:
cond_seq = torch.zeros(generation_length, num_cond_params)

num_cond_params = model_config.cond_size
cond_seq = torch.zeros(generation_length, num_cond_params)

# instID
# cond_seq[:, 0] = 0
# cond_seq[:, 1] = 0
# cond_seq[:, 2] = 0   #bugs
# cond_seq[:, 3] = 0
# cond_seq[:, 4] = 0   #pistons
# cond_seq[:, 5] = 0
# cond_seq[:, 6] = 1

# param_1
#cond_seq[:, 0] = torch.FloatTensor(multi_linspace([(0, 0),(.2,0), (.3,.7), (.5,.7),(.75,0),(1,0)], generation_length))
cond_seq[:, 0] = torch.FloatTensor(multi_linspace([(0, 0),(.05,0), (.45, 1), (.55,1),(.95,0),(1,0)], generation_length))

plot_condition_tensor(cond_seq, frame_rate)

# generated_audio_glide, sr = run_inference(model, enc_model, cond_seq.to(device), warmup_sequence.to(device), clamp_val, top_n=top_n, temperature=temperature, include_warmup_audio=False)

start = time.perf_counter()
generated_audio_glide=bazrnn.getNextAudioHop(cond_seq.to(device)) 
elapsed = time.perf_counter() - start
print(f"Generated { generated_audio.shape[0]} samples  in {elapsed:.2f}s.")


# Create a time axis in seconds
t = np.arange(len(generated_audio_glide)) / sr

plt.figure(figsize=(20, 5))
plt.plot(t, generated_audio_glide)
plt.title("Generated Audio Waveform")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.grid()

plt.show()
Audio(generated_audio_glide, rate=sr)

<div style="width: 25%; height: 6px; background-color: blue;"></div>
<b>Just another visualization of the above</b>

In [ ]:
# Create time axes
cond_time = np.linspace(0, len(generated_audio_glide)/sr, len(cond_seq))
audio_time = np.arange(len(generated_audio_glide)) / sr

fig, ax1 = plt.subplots(figsize=(20, 4))

# Plot audio waveform
color1 = '#0000AA'
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Audio Amplitude', color=color1)
ax1.plot(audio_time, generated_audio_glide, color=color1, alpha=0.6, label='Audio')
ax1.tick_params(axis='y', labelcolor=color1)

# Create second y-axis for conditions
color2 = '#880022'
ax2 = ax1.twinx()
ax2.set_ylabel('Condition Value', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)


# Plot all condition parameters with different colors
colors = ['#AA0022', 'tab:orange', 'tab:green', 'tab:purple', 'tab:brown', 'tab:pink', 'tab:gray', 'tab:olive', 'tab:cyan']

for i in range(cond_seq.shape[1]):  # Loop through all condition parameters
    color = colors[i % len(colors)]  # Cycle through colors if more params than colors
    ax2.plot(cond_time, cond_seq[:, i], color=color, linewidth=1.5, alpha=.8, label=f'Param {i+1}')

# ax2.tick_params(axis='y', labelcolor='black')

# Add legends
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

plt.title("Generated Audio Waveform with Condition Parameters")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()